In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import time

# ────────────────────────────────────────────────
# 1. Load and prepare MNIST (simple grayscale dataset)
# ────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

x_train = np.expand_dims(x_train, -1)   # (N,28,28,1)
x_test  = np.expand_dims(x_test, -1)

y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

print("Data ready. Shape:", x_train.shape)

Data ready. Shape: (60000, 28, 28, 1)


In [2]:
def build_cnn(padding='valid', strides=(1,1), kernel_size=3):
    model = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        
        layers.Conv2D(32, kernel_size, 
                      padding=padding, 
                      strides=strides,
                      activation='relu'),
        layers.MaxPooling2D(pool_size=(2,2)),
        
        layers.Conv2D(64, kernel_size, 
                      padding=padding, 
                      strides=strides,
                      activation='relu'),
        layers.MaxPooling2D(pool_size=(2,2)),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

In [3]:
experiments = [
    {"name": "Baseline (same padding, stride 1)",     "padding": "same",  "strides": (1,1)},
    {"name": "No padding (valid), stride 1",          "padding": "valid", "strides": (1,1)},
    {"name": "Same padding, stride 2",                "padding": "same",  "strides": (2,2)},
    {"name": "Valid padding, stride 2",               "padding": "valid", "strides": (2,2)},
    {"name": "Same padding, stride 3",                "padding": "same",  "strides": (3,3)},
]

results = []

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running experiment: {exp['name']}")
    
    model = build_cnn(padding=exp["padding"], strides=exp["strides"])
    model.summary()
    
    start_time = time.time()
    
    history = model.fit(x_train, y_train,
                        batch_size=128,
                        epochs=8,           # keep short for lab time
                        validation_split=0.1,
                        verbose=1)
    
    train_time = time.time() - start_time
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    
    results.append({
        "name": exp["name"],
        "padding": exp["padding"],
        "strides": exp["strides"],
        "train_time_sec": round(train_time, 1),
        "test_accuracy": round(test_acc, 4),
        "final_val_acc": round(history.history['val_accuracy'][-1], 4),
        "num_params": model.count_params()
    })


Running experiment: Baseline (same padding, stride 1)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 10, 10, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 2, 2, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 1, 1, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,426 (111.04 KB)

 Trainable params: 28,426 (111.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7540 - loss: 0.7623 - val_accuracy: 0.9222 - val_loss: 0.2613
Epoch 2/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9121 - loss: 0.2849 - val_accuracy: 0.9490 - val_loss: 0.1709
Epoch 3/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9350 - loss: 0.2078 - val_accuracy: 0.9572 - val_loss: 0.1377
Epoch 4/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9460 - loss: 0.1722 - val_accuracy: 0.9608 - val_loss: 0.1256
Epoch 5/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9534 - loss: 0.1510 - val_accuracy: 0.9653 - val_loss: 0.1167
Epoch 6/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9586 - loss: 0.1327 - val_accuracy: 0.9678 - val_loss: 0.1039
Epoch 7/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9621 - loss: 0.1200 - val_accuracy: 0.9710 - val_loss: 0.0957
Epoch 8/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9648 - loss: 0.1120 - val_accuracy: 0.

In [5]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("-"*80)
print(f"{'Config':<35} {'Padding':<10} {'Stride':<8} {'Params':<12} {'Train Time':<12} {'Test Acc':<10}")
print("-"*80)
for r in results:
    print(f"{r['name']:<35} {r['padding']:<10} {str(r['strides']):<8} {r['num_params']:<12,} {r['train_time_sec']:<12} {r['test_accuracy']:<10.4f}")
print("="*80)

print("\nKey observations to discuss:")
print("• 'same' padding usually gives slightly higher accuracy (preserves edges)")
print("• Higher strides → much fewer parameters and faster training, but accuracy drops")
print("• Trade-off becomes more obvious on harder datasets (try CIFAR-10 next)")


EXPERIMENT SUMMARY
--------------------------------------------------------------------------------
Config                              Padding    Stride   Params       Train Time   Test Acc  
--------------------------------------------------------------------------------
Baseline (same padding, stride 1)   same       (1, 1)   421,642      162.7        0.9913    
No padding (valid), stride 1        valid      (1, 1)   225,034      121.2        0.9916    
Same padding, stride 2              same       (2, 2)   53,002       53.6         0.9867    
Valid padding, stride 2             valid      (2, 2)   28,426       42.2         0.9730    
Same padding, stride 3              same       (3, 3)   28,426       38.6         0.9679    

Key observations to discuss:
• 'same' padding usually gives slightly higher accuracy (preserves edges)
• Higher strides → much fewer parameters and faster training, but accuracy drops
• Trade-off becomes more obvious on harder datasets (try CIFAR-10 next)
